In [9]:
# from pyspark.sql import SparkSession

# spark = SparkSession.builder \
#     .appName("Read Multiple Excel Files") \
#     .master("local[*]") \
#     .config("spark.driver.memory", "24g") \
#     .config("spark.memory.offHeap.enabled", "true") \
#     .config("spark.memory.offHeap.size", "4g") \
#     .getOrCreate()


25/04/29 19:18:02 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [10]:
# df = spark.read.csv("2024_lca_combined_df.csv", header=True)

# Data Exploration

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
from sklearn.feature_selection import VarianceThreshold

In [3]:
lca_df = pd.read_csv("lca_disclosure_data_2024_total.csv")

In [10]:
# Top 10 most frequent H1-B Applications
lca_df.groupby(["EMPLOYER_NAME", "JOB_TITLE"])["CASE_NUMBER"].count().sort_values(ascending=False).head(10)

EMPLOYER_NAME                           JOB_TITLE                                                   
Google LLC                              Software Engineer                                               5581
COGNIZANT TECHNOLOGY SOLUTIONS US CORP  Manager JC50 - Computer Systems Engineers/Architects            4335
Microsoft Corporation                   Software Engineering                                            3743
COGNIZANT TECHNOLOGY SOLUTIONS US CORP  Senior Associate JC60 -Computer Systems Engineers/Architects    2574
Microsoft Corporation                   Software Engineer                                               2284
Amazon.com Services LLC                 Software Development Engineer II                                2119
TATA CONSULTANCY SERVICES LIMITED       Architect                                                       1571
Meta Platforms, Inc                     Software Engineer                                               1476
Tata Consultancy Services L

## Predict actual wage and compare it to prevailing wage 

In [3]:
lca_df.shape

(561037, 98)

In [4]:
# Use get_dummies() function for dummy encoding
dummy_df = pd.get_dummies(lca_df[['SOC_TITLE', 'JOB_TITLE', 'FULL_TIME_POSITION', 'VISA_CLASS']], sparse=True)

# Concatenate the dummy dataframe with the original dataframe
lca_df = pd.concat([lca_df, dummy_df], axis=1)

In [4]:
dummy_df.head()

,SOC_TITLE_15-1299.08 Computer Systems Engineers/Architects,SOC_TITLE_ACCOUNTANTS AND AUDITORS,SOC_TITLE_AEROSPACE ENGINEERS,"SOC_TITLE_ARCHITECTURE TEACHERS, POSTSECONDARY",SOC_TITLE_ART DIRECTOR,SOC_TITLE_ART DIRECTORS,SOC_TITLE_Accountants,SOC_TITLE_Accountants and Auditors,SOC_TITLE_Accountants anf Auditors,SOC_TITLE_Actuaries,...,JOB_TITLE_¿¿¿Project Manager,JOB_TITLE_– Sr. Engineering Manager,JOB_TITLE_‘Software Developer / ServiceNow Developer’,JOB_TITLE_•\tPrincipal Solutions Architect,FULL_TIME_POSITION_N,FULL_TIME_POSITION_Y,VISA_CLASS_E-3 Australian,VISA_CLASS_H-1B,VISA_CLASS_H-1B1 Chile,VISA_CLASS_H-1B1 Singapore
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,True,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,True,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,True,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,True,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,True,False,False


In [ ]:
dummy_df.corr()

In [19]:
X,y = dummy_df, lca_df["WAGE_RATE_OF_PAY_FROM"]

X_train, y_train, X_test, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

lin_model = LinearRegression().fit(X_train, y_train, n_jobs=-1)
linear_predictions = lin_model.predict(X_test)

KeyboardInterrupt: 

In [ ]:
print(f"MSE: {round(mean_squared_error(y_test, linear_predictions),2)}")

In [20]:
lca_df.shape

(561037, 121374)